## Cell 1 - Setup
Fresh kernel — no stale class definitions.

In [ ]:
!pip install timm diffusers==0.30.3 accelerate fvcore scikit-image -q
import os, sys
if not os.path.exists('DiT'):
    !git clone https://github.com/facebookresearch/DiT.git -q
sys.path.insert(0, os.path.abspath('DiT'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from fvcore.nn import FlopCountAnalysis
from models import DiT_models

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    print(f'PyTorch: {torch.__version__}')


## Cell 2 - Load DiT-XL/2
Load to CPU first to avoid SM 12.0 kernel mismatch, then move to GPU.

In [ ]:
CKPT = 'DiT-XL-2-256x256.pt'
if not os.path.exists(CKPT):
    !wget -q --show-progress https://dl.fbaipublicfiles.com/DiT/models/DiT-XL-2-256x256.pt

# patch models.py for BF16 compatibility
import DiT.models as dit_models
_orig_te = dit_models.TimestepEmbedder.forward
def _patched_te(self, t):
    t_freq = self.timestep_embedding(t, self.frequency_embedding_size)
    t_freq = t_freq.to(self.mlp[0].weight.dtype)
    return self.mlp(t_freq)
dit_models.TimestepEmbedder.forward = _patched_te

_orig_dit_fwd = dit_models.DiT.forward
def _patched_dit_fwd(self, x, t, y):
    x = x.to(self.x_embedder.proj.weight.dtype)
    return _orig_dit_fwd(self, x, t, y)
dit_models.DiT.forward = _patched_dit_fwd

# load to CPU first then GPU
dit = DiT_models['DiT-XL/2'](input_size=32)
state = torch.load(CKPT, map_location='cpu')
dit.load_state_dict(state.get('model', state))
dit = dit.float().to(device)

for p in dit.parameters():
    p.requires_grad = False
dit.eval()

hidden_dim  = dit.x_embedder.proj.out_channels
IN_CHANNELS = 4
model_dtype = next(dit.parameters()).dtype

print(f'DiT loaded | dtype={model_dtype} | hidden_dim={hidden_dim}')
print(f'Frozen: {not any(p.requires_grad for p in dit.parameters())}')


## Cell 3 - OLD PruningWrapper (Python loop)
Baseline implementation for comparison.

In [ ]:
def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

def _run_attn_half(block, x, c):
    shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = \
        block.adaLN_modulation(c).chunk(6, dim=1)
    attn_out    = block.attn(modulate(block.norm1(x), shift_msa, scale_msa))
    x_post_attn = x + gate_msa.unsqueeze(1) * attn_out
    return x_post_attn, (shift_mlp, scale_mlp, gate_mlp)

def _run_mlp_half(block, x, mlp_params):
    shift_mlp, scale_mlp, gate_mlp = mlp_params
    mlp_out = block.mlp(modulate(block.norm2(x), shift_mlp, scale_mlp))
    return x + gate_mlp.unsqueeze(1) * mlp_out

class TokenPredictor(nn.Module):
    def __init__(self, d_model, bottleneck=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, bottleneck),
            nn.GELU(), nn.Linear(bottleneck, 1),
        )
    def forward(self, x, temperature=1.0):
        logits = self.mlp(x)
        pair   = torch.cat([logits, -logits], dim=-1)
        hard   = F.gumbel_softmax(pair, tau=temperature, hard=True)
        return hard[..., 1], logits

class PruningWrapperOLD(nn.Module):
    '''Original Python-loop based gather/scatter.'''
    def __init__(self, block, predictor):
        super().__init__()
        self.block           = block
        self.predictor       = predictor
        self.pruning_enabled = True
        self._last_mask      = None
        self._live_mask      = None
        self._temperature    = 1.0

    def forward(self, x, c):
        if not self.pruning_enabled:
            self._live_mask = None
            return self.block(x, c)
        B, N, D = x.shape
        x_post_attn, mlp_params = _run_attn_half(self.block, x, c)
        keep_mask, _ = self.predictor(x_post_attn, self._temperature)
        self._live_mask = keep_mask
        self._last_mask = keep_mask.detach()
        out = torch.zeros_like(x_post_attn)
        for b in range(B):
            idx  = keep_mask[b].detach().bool()
            N_k  = idx.sum().item()
            mp_b = tuple(p[b].unsqueeze(0) for p in mlp_params)
            if N_k == 0 or N_k == N:
                out[b] = _run_mlp_half(self.block, x_post_attn[b].unsqueeze(0), mp_b).squeeze(0)
            else:
                ii = idx.nonzero(as_tuple=True)[0]
                out_k = _run_mlp_half(self.block, x_post_attn[b,ii,:].unsqueeze(0), mp_b).squeeze(0)
                out_b = x_post_attn[b].clone()
                out_b[ii] = out_k
                out[b] = out_b
        return out

class PruningWrapperNEW(nn.Module):
    '''Batched sparse gather/scatter — no Python loop, no .item() calls.'''
    def __init__(self, block, predictor):
        super().__init__()
        self.block           = block
        self.predictor       = predictor
        self.pruning_enabled = True
        self._last_mask      = None
        self._live_mask      = None
        self._temperature    = 1.0

    def forward(self, x, c):
        if not self.pruning_enabled:
            self._live_mask = None
            return self.block(x, c)
        B, N, D = x.shape
        x_post_attn, (shift_mlp, scale_mlp, gate_mlp) = \
            _run_attn_half(self.block, x, c)
        keep_mask, _ = self.predictor(x_post_attn, self._temperature)
        self._live_mask = keep_mask
        self._last_mask = keep_mask.detach()
        mask_flat = keep_mask.detach().bool().reshape(B * N)
        x_flat    = x_post_attn.reshape(B * N, D)
        N_kept    = mask_flat.sum().item()
        if N_kept == 0:
            return x_post_attn
        if N_kept == B * N:
            norm_x  = self.block.norm2(x_post_attn)
            mlp_in  = norm_x * (1 + scale_mlp.unsqueeze(1)) + shift_mlp.unsqueeze(1)
            mlp_out = self.block.mlp(mlp_in)
            return x_post_attn + gate_mlp.unsqueeze(1) * mlp_out
        x_kept     = x_flat[mask_flat]
        shift_flat = shift_mlp.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1)[mask_flat]
        scale_flat = scale_mlp.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1)[mask_flat]
        gate_flat  = gate_mlp.unsqueeze(1).expand(B,N,-1).reshape(B*N,-1)[mask_flat]
        norm_kept  = self.block.norm2(x_kept)
        mlp_in     = norm_kept * (1 + scale_flat) + shift_flat
        mlp_out    = self.block.mlp(mlp_in)
        out_kept   = x_kept + gate_flat * mlp_out
        out_flat   = x_flat.clone()
        out_flat[mask_flat] = out_kept
        return out_flat.reshape(B, N, D)

def set_pruning_mode(model, enabled):
    for b in model.blocks:
        if isinstance(b, (PruningWrapperOLD, PruningWrapperNEW)):
            b.pruning_enabled = enabled

print('Both PruningWrapper versions defined cleanly in fresh kernel.')
print('  PruningWrapperOLD: Python loop')
print('  PruningWrapperNEW: Batched sparse gather')


## Cell 4 - Benchmark Function

In [ ]:
PRUNING_LAYERS = list(range(0, 28, 2))
REPS   = 50
WARMUP = 10

def wrap_model(WrapperClass):
    '''Peel all wrappers then rewrap with the given class.'''
    for idx in PRUNING_LAYERS:
        b = dit.blocks[idx]
        # peel any existing wrapper of any class
        while hasattr(b, 'block') and hasattr(b, 'predictor'):
            b = b.block
        dit.blocks[idx] = b
    for idx in PRUNING_LAYERS:
        pred = TokenPredictor(hidden_dim, 64).float().to(device)
        dit.blocks[idx] = WrapperClass(dit.blocks[idx], pred)
    outer = type(dit.blocks[0]).__name__
    inner = type(dit.blocks[0].block).__name__
    print(f'Wrapped: {outer} -> {inner}')

def benchmark_batch(batch_size):
    z = torch.randn(batch_size, 4, 32, 32, device=device, dtype=torch.float32)
    t = torch.randint(0, 1000, (batch_size,), device=device)
    y = torch.zeros(batch_size, dtype=torch.long, device=device)
    starter = torch.cuda.Event(enable_timing=True)
    ender   = torch.cuda.Event(enable_timing=True)

    set_pruning_mode(dit, enabled=False)
    dit.eval()
    with torch.no_grad():
        for _ in range(WARMUP): _ = dit(z, t, y)
        torch.cuda.synchronize()
        starter.record()
        for _ in range(REPS): _ = dit(z, t, y)
        ender.record()
    torch.cuda.synchronize()
    ms_t = starter.elapsed_time(ender) / REPS

    set_pruning_mode(dit, enabled=True)
    with torch.no_grad():
        for _ in range(WARMUP): _ = dit(z, t, y)
        torch.cuda.synchronize()
        starter.record()
        for _ in range(REPS): _ = dit(z, t, y)
        ender.record()
    torch.cuda.synchronize()
    ms_s = starter.elapsed_time(ender) / REPS

    return ms_t, ms_s

BATCH_SIZES = [1, 4, 8, 16, 32, 64]
print('Benchmark ready.')


## Cell 5 - Benchmark: OLD Python Loop

In [ ]:
print('Wrapping with OLD PruningWrapper (Python loop)...')
wrap_model(PruningWrapperOLD)

results_old = []
print('\nBatch | Teacher ms | Student ms | Speedup  | Status')
print('-' * 60)
for bs in BATCH_SIZES:
    ms_t, ms_s = benchmark_batch(bs)
    speedup = ms_t / ms_s
    status  = 'FASTER' if speedup > 1.0 else 'slower'
    results_old.append({'batch': bs, 'ms_t': ms_t, 'ms_s': ms_s, 'speedup': speedup})
    print(f'{bs:5d} | {ms_t:10.2f} | {ms_s:10.2f} | {speedup:8.3f}x | {status}')


## Cell 6 - Benchmark: NEW Batched Sparse Gather

In [ ]:
print('Wrapping with NEW PruningWrapper (batched sparse)...')
wrap_model(PruningWrapperNEW)

results_new = []
print('\nBatch | Teacher ms | Student ms | Speedup  | Status')
print('-' * 60)
for bs in BATCH_SIZES:
    ms_t, ms_s = benchmark_batch(bs)
    speedup = ms_t / ms_s
    status  = 'FASTER' if speedup > 1.0 else 'slower'
    results_new.append({'batch': bs, 'ms_t': ms_t, 'ms_s': ms_s, 'speedup': speedup})
    print(f'{bs:5d} | {ms_t:10.2f} | {ms_s:10.2f} | {speedup:8.3f}x | {status}')


## Cell 7 - GMACs via fvcore

In [ ]:
print('Computing GMACs (batch=1, NEW wrapper)...')
wrap_model(PruningWrapperNEW)
z1 = torch.randn(1, 4, 32, 32, device=device, dtype=torch.float32)
t1 = torch.randint(0, 1000, (1,), device=device)
y1 = torch.zeros(1, dtype=torch.long, device=device)

try:
    set_pruning_mode(dit, enabled=False)
    fa = FlopCountAnalysis(dit, (z1, t1, y1))
    fa.unsupported_ops_warnings(False)
    gmac_t = fa.total() / 1e9

    set_pruning_mode(dit, enabled=True)
    fb = FlopCountAnalysis(dit, (z1, t1, y1))
    fb.unsupported_ops_warnings(False)
    gmac_s = fb.total() / 1e9

    reduction = (1 - gmac_s / gmac_t) * 100
    print(f'Teacher GMACs : {gmac_t:.2f}')
    print(f'Student GMACs : {gmac_s:.2f}')
    print(f'Reduction     : {reduction:.1f}%')
except Exception as e:
    gmac_t = gmac_s = reduction = 0.0
    print(f'fvcore failed: {e}')


## Cell 8 - Plot: OLD vs NEW comparison

In [ ]:
import numpy as np

batches      = [r['batch']   for r in results_old]
speedups_old = [r['speedup'] for r in results_old]
speedups_new = [r['speedup'] for r in results_new]
ms_t_old     = [r['ms_t']   for r in results_old]
ms_s_old     = [r['ms_s']   for r in results_old]
ms_t_new     = [r['ms_t']   for r in results_new]
ms_s_new     = [r['ms_s']   for r in results_new]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# speedup comparison
axes[0].plot(batches, speedups_old, 'o--', color='#E24B4A', linewidth=2,
             markersize=8, label='OLD (Python loop)')
axes[0].plot(batches, speedups_new, 'o-',  color='#1D9E75', linewidth=2,
             markersize=8, label='NEW (batched sparse)')
axes[0].axhline(1.0, color='#EF9F27', linestyle='--', linewidth=1.5, label='Breakeven')
axes[0].set_xlabel('Batch size'); axes[0].set_ylabel('Speedup (teacher/student)')
axes[0].set_title('Speedup: OLD vs NEW'); axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log', base=2)
axes[0].set_xticks(batches); axes[0].set_xticklabels([str(b) for b in batches])

# latency OLD
x = np.arange(len(batches)); w = 0.35
axes[1].bar(x-w/2, ms_t_old, w, label='Teacher', color='#378ADD', alpha=0.85)
axes[1].bar(x+w/2, ms_s_old, w, label='Student OLD', color='#E24B4A', alpha=0.85)
axes[1].set_xlabel('Batch size'); axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency: OLD implementation')
axes[1].set_xticks(x); axes[1].set_xticklabels([str(b) for b in batches])
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')

# latency NEW
axes[2].bar(x-w/2, ms_t_new, w, label='Teacher',     color='#378ADD', alpha=0.85)
axes[2].bar(x+w/2, ms_s_new, w, label='Student NEW', color='#1D9E75', alpha=0.85)
axes[2].set_xlabel('Batch size'); axes[2].set_ylabel('Latency (ms)')
axes[2].set_title('Latency: NEW batched implementation')
axes[2].set_xticks(x); axes[2].set_xticklabels([str(b) for b in batches])
axes[2].legend(); axes[2].grid(True, alpha=0.3, axis='y')

title_str = (
    f'Teacher: {gmac_t:.2f} GMACs  |  '
    f'Student: {gmac_s:.2f} GMACs  |  '
    f'Reduction: {reduction:.1f}%'
) if gmac_t else 'FLOP Benchmark: OLD vs NEW PruningWrapper'
plt.suptitle(title_str, fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('benchmark_old_vs_new.png', dpi=150, bbox_inches='tight')
plt.show()

# summary table
print('\n' + '=' * 72)
print(f'  Batch  | OLD speedup | NEW speedup | Improvement')
print('=' * 72)
for ro, rn in zip(results_old, results_new):
    improvement = rn['speedup'] - ro['speedup']
    arrow = '+' if improvement > 0 else ''
    bl = str(ro['batch'])
    print(f'  {bl:<6} | {ro["speedup"]:>11.3f}x | {rn["speedup"]:>11.3f}x | {arrow}{improvement:+.3f}x')
print('=' * 72)
crossover_new = next((r['batch'] for r in results_new if r['speedup'] > 1.0), None)
crossover_old = next((r['batch'] for r in results_old if r['speedup'] > 1.0), None)
print(f'\nOLD crossover: {crossover_old if crossover_old else "not reached"}')
print(f'NEW crossover: {crossover_new if crossover_new else "not reached"}')
print('\nSaved: benchmark_old_vs_new.png')
